In [1]:
123

123

In [ ]:
# -----------------------------------------------------------------------------
# 환경 설정: SparkSession 생성 및 데이터 로드
# -----------------------------------------------------------------------------
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, upper, lower, concat, substring
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, DateType
import pandas as pd
import numpy as np
import os

# SparkSession 생성 (이전 교시에서 이미 있으면 재사용)
spark = SparkSession.builder \
    .appName("PySpark-Column-Filter") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", 10) \
    .getOrCreate()

# 테스트 데이터 생성 (이전 교시 데이터 없으면 새로 생성)
np.random.seed(42)

os.makedirs("/tmp/spark_tutorial", exist_ok=True)

# 샘플 데이터
sample_data = pd.DataFrame({
    "emp_id": [f"E{i:03d}" for i in range(1, 101)],
    "name": [f"Employee_{i}" for i in range(1, 101)],
    "department": np.random.choice(
        ["Engineering", "Sales", "Marketing", "HR", "Finance"], 100
    ),
    "salary": np.random.randint(40000, 120000, 100),
    "age": np.random.randint(25, 55, 100),
    "is_manager": np.random.choice([True, False], 100, p=[0.2, 0.8]),
})

# 결측치 추가
sample_data.loc[5:10, "salary"] = None
sample_data.loc[15:18, "department"] = None

sample_data.to_csv("/tmp/spark_tutorial/employees.csv", index=False)

# DataFrame 로드
df = spark.read.csv("/tmp/spark_tutorial/employees.csv", header=True, inferSchema=True)

print("데이터 로드 완료!")
print(f"행 수: {df.count()}, 컬럼: {df.columns}")
df.show(5)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/20 00:25:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


데이터 로드 완료!
행 수: 100, 컬럼: ['emp_id', 'name', 'department', 'salary', 'age', 'is_manager']
+------+----------+----------+--------+---+----------+
|emp_id|      name|department|  salary|age|is_manager|
+------+----------+----------+--------+---+----------+
|  E001|Employee_1|        HR| 92251.0| 28|     false|
|  E002|Employee_2|   Finance| 62662.0| 43|     false|
|  E003|Employee_3| Marketing| 48392.0| 50|     false|
|  E004|Employee_4|   Finance| 70535.0| 27|     false|
|  E005|Employee_5|   Finance|118603.0| 43|     false|
+------+----------+----------+--------+---+----------+
only showing top 5 rows



26/01/20 00:25:53 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


### < 컬럼 참조 방법 비교 >

In [3]:
from pyspark.sql.functions import col

In [4]:
# 방법 1: 문자열
df.select("name","salary").show(5)

+----------+--------+
|      name|  salary|
+----------+--------+
|Employee_1| 92251.0|
|Employee_2| 62662.0|
|Employee_3| 48392.0|
|Employee_4| 70535.0|
|Employee_5|118603.0|
+----------+--------+
only showing top 5 rows



In [5]:
# 방법 2: col() 함수 -> 연산, 조건식 사용 가능
df.select(col("name"), col("salary")*1.1).show(5)

+----------+------------------+
|      name|    (salary * 1.1)|
+----------+------------------+
|Employee_1|          101476.1|
|Employee_2| 68928.20000000001|
|Employee_3|53231.200000000004|
|Employee_4|           77588.5|
|Employee_5|130463.30000000002|
+----------+------------------+
only showing top 5 rows



In [7]:
# 방법 3: df.컬럼명 -> 조인 시 유용
df.select(df.name, df.salary).show(5)

+----------+--------+
|      name|  salary|
+----------+--------+
|Employee_1| 92251.0|
|Employee_2| 62662.0|
|Employee_3| 48392.0|
|Employee_4| 70535.0|
|Employee_5|118603.0|
+----------+--------+
only showing top 5 rows



In [8]:
df.show(5)

+------+----------+----------+--------+---+----------+
|emp_id|      name|department|  salary|age|is_manager|
+------+----------+----------+--------+---+----------+
|  E001|Employee_1|        HR| 92251.0| 28|     false|
|  E002|Employee_2|   Finance| 62662.0| 43|     false|
|  E003|Employee_3| Marketing| 48392.0| 50|     false|
|  E004|Employee_4|   Finance| 70535.0| 27|     false|
|  E005|Employee_5|   Finance|118603.0| 43|     false|
+------+----------+----------+--------+---+----------+
only showing top 5 rows



In [9]:
# 문자열은 연산 불가 (에러 발생)
# df.select("salary" * 1.1)  # TypeError!

# col()은 연산 가능
df.select(
    col("name"),
    col("salary"),
    col("salary") * 1.1
).show(5)

+----------+--------+------------------+
|      name|  salary|    (salary * 1.1)|
+----------+--------+------------------+
|Employee_1| 92251.0|          101476.1|
|Employee_2| 62662.0| 68928.20000000001|
|Employee_3| 48392.0|53231.200000000004|
|Employee_4| 70535.0|           77588.5|
|Employee_5|118603.0|130463.30000000002|
+----------+--------+------------------+
only showing top 5 rows



### 실습 1

In [12]:
# 테이블 확인
df.show(5)

+------+----------+----------+--------+---+----------+
|emp_id|      name|department|  salary|age|is_manager|
+------+----------+----------+--------+---+----------+
|  E001|Employee_1|        HR| 92251.0| 28|     false|
|  E002|Employee_2|   Finance| 62662.0| 43|     false|
|  E003|Employee_3| Marketing| 48392.0| 50|     false|
|  E004|Employee_4|   Finance| 70535.0| 27|     false|
|  E005|Employee_5|   Finance|118603.0| 43|     false|
+------+----------+----------+--------+---+----------+
only showing top 5 rows



In [14]:
# name과 age 컬럼을 col() 함수 사용해서 선택
df.select(col("name"), col("age")).show(5)

+----------+---+
|      name|age|
+----------+---+
|Employee_1| 28|
|Employee_2| 43|
|Employee_3| 50|
|Employee_4| 27|
|Employee_5| 43|
+----------+---+
only showing top 5 rows



In [15]:
# salary 컬럼에 2를 곱한 결과를 조회
df.select(col("salary")*2).show(5)

+------------+
|(salary * 2)|
+------------+
|    184502.0|
|    125324.0|
|     96784.0|
|    141070.0|
|    237206.0|
+------------+
only showing top 5 rows



In [16]:
# df.name과 df.department를 사용해 두 컬럼 조회
df.select(df.name, df.department).show(5)

+----------+----------+
|      name|department|
+----------+----------+
|Employee_1|        HR|
|Employee_2|   Finance|
|Employee_3| Marketing|
|Employee_4|   Finance|
|Employee_5|   Finance|
+----------+----------+
only showing top 5 rows



### 컬럼 선택(select)

In [17]:
# 기본 형태
df.select("name").show(5)

+----------+
|      name|
+----------+
|Employee_1|
|Employee_2|
|Employee_3|
|Employee_4|
|Employee_5|
+----------+
only showing top 5 rows



In [ ]:
# 여러 컬럼 선택
df.select("emp_id","name","department").show(5)

+------+----------+----------+
|emp_id|      name|department|
+------+----------+----------+
|  E001|Employee_1|        HR|
|  E002|Employee_2|   Finance|
|  E003|Employee_3| Marketing|
|  E004|Employee_4|   Finance|
|  E005|Employee_5|   Finance|
+------+----------+----------+
only showing top 5 rows



In [21]:
# 컬럼 연산과 별칭(alias)
result = df.select(
    col("name"),
    col("salary"),
    (col("salary")*1.1).alias("raised_salary"),
    (col("salary") / 12).alias("monthly_salary")
)
print("=== 급여 계산 ===")
result.show(5)

=== 급여 계산 ===
+----------+--------+------------------+------------------+
|      name|  salary|     raised_salary|    monthly_salary|
+----------+--------+------------------+------------------+
|Employee_1| 92251.0|          101476.1| 7687.583333333333|
|Employee_2| 62662.0| 68928.20000000001| 5221.833333333333|
|Employee_3| 48392.0|53231.200000000004|4032.6666666666665|
|Employee_4| 70535.0|           77588.5| 5877.916666666667|
|Employee_5|118603.0|130463.30000000002| 9883.583333333334|
+----------+--------+------------------+------------------+
only showing top 5 rows



In [22]:
# 컬럼 순서 변경
reordered = df.select("department", "name", "salary", "emp_id")
print("=== 컬럼 순서 변경 ===")
reordered.show(3)

=== 컬럼 순서 변경 ===
+----------+----------+-------+------+
|department|      name| salary|emp_id|
+----------+----------+-------+------+
|        HR|Employee_1|92251.0|  E001|
|   Finance|Employee_2|62662.0|  E002|
| Marketing|Employee_3|48392.0|  E003|
+----------+----------+-------+------+
only showing top 3 rows



In [23]:
# 원래 컬럼 순서
df.show(3)

+------+----------+----------+-------+---+----------+
|emp_id|      name|department| salary|age|is_manager|
+------+----------+----------+-------+---+----------+
|  E001|Employee_1|        HR|92251.0| 28|     false|
|  E002|Employee_2|   Finance|62662.0| 43|     false|
|  E003|Employee_3| Marketing|48392.0| 50|     false|
+------+----------+----------+-------+---+----------+
only showing top 3 rows



In [24]:
# 특정 컬럼 제외 : drop() 사용
# drop() : 지정한 컬럼을 제외한 나머지 반환
without_manager = df.drop("is_manager")
print("=== is_manager 컬럼 제외 ===")
without_manager.show(5)

=== is_manager 컬럼 제외 ===
+------+----------+----------+--------+---+
|emp_id|      name|department|  salary|age|
+------+----------+----------+--------+---+
|  E001|Employee_1|        HR| 92251.0| 28|
|  E002|Employee_2|   Finance| 62662.0| 43|
|  E003|Employee_3| Marketing| 48392.0| 50|
|  E004|Employee_4|   Finance| 70535.0| 27|
|  E005|Employee_5|   Finance|118603.0| 43|
+------+----------+----------+--------+---+
only showing top 5 rows



In [26]:
# 동적 컬럼 선택 (리스트)

# 컬럼 목록을 변수로 관리
cols_to_select = ["emp_id", "name", "salary"]

df.select(cols_to_select).show(5)

+------+----------+--------+
|emp_id|      name|  salary|
+------+----------+--------+
|  E001|Employee_1| 92251.0|
|  E002|Employee_2| 62662.0|
|  E003|Employee_3| 48392.0|
|  E004|Employee_4| 70535.0|
|  E005|Employee_5|118603.0|
+------+----------+--------+
only showing top 5 rows



In [27]:
# *연산자로 리스트 언패킹
df.select(*cols_to_select).show(5)

+------+----------+--------+
|emp_id|      name|  salary|
+------+----------+--------+
|  E001|Employee_1| 92251.0|
|  E002|Employee_2| 62662.0|
|  E003|Employee_3| 48392.0|
|  E004|Employee_4| 70535.0|
|  E005|Employee_5|118603.0|
+------+----------+--------+
only showing top 5 rows



In [28]:
# 조건에 따라 컬럼 선택
numeric_cols = ["salary", "age"]
df.select(*numeric_cols).describe().show()

+-------+------------------+-----------------+
|summary|            salary|              age|
+-------+------------------+-----------------+
|  count|                94|              100|
|   mean| 80864.46808510639|            38.83|
| stddev|22171.614668892573|9.084402216786948|
|    min|           40206.0|               25|
|    max|          119309.0|               54|
+-------+------------------+-----------------+



### 실습 2

In [29]:
df.show(5)

+------+----------+----------+--------+---+----------+
|emp_id|      name|department|  salary|age|is_manager|
+------+----------+----------+--------+---+----------+
|  E001|Employee_1|        HR| 92251.0| 28|     false|
|  E002|Employee_2|   Finance| 62662.0| 43|     false|
|  E003|Employee_3| Marketing| 48392.0| 50|     false|
|  E004|Employee_4|   Finance| 70535.0| 27|     false|
|  E005|Employee_5|   Finance|118603.0| 43|     false|
+------+----------+----------+--------+---+----------+
only showing top 5 rows



In [ ]:
# emp_id, name, salary 3개 컬럼만 선택
df.select("emp_id","name","salary").show(5)

+------+----------+--------+
|emp_id|      name|  salary|
+------+----------+--------+
|  E001|Employee_1| 92251.0|
|  E002|Employee_2| 62662.0|
|  E003|Employee_3| 48392.0|
|  E004|Employee_4| 70535.0|
|  E005|Employee_5|118603.0|
+------+----------+--------+
only showing top 5 rows



In [32]:
# salary 컬럼을 1000으로 나눈 값을 "salary_k"라는 이름으로 조회
df.select(
    (col("salary") / 1000).alias("salary_k")
).show(5)

+--------+
|salary_k|
+--------+
|  92.251|
|  62.662|
|  48.392|
|  70.535|
| 118.603|
+--------+
only showing top 5 rows



In [33]:
# is_manager 컬럼을 제외한 나머지 컬럼을 조회
df.drop("is_manager").show(5)

+------+----------+----------+--------+---+
|emp_id|      name|department|  salary|age|
+------+----------+----------+--------+---+
|  E001|Employee_1|        HR| 92251.0| 28|
|  E002|Employee_2|   Finance| 62662.0| 43|
|  E003|Employee_3| Marketing| 48392.0| 50|
|  E004|Employee_4|   Finance| 70535.0| 27|
|  E005|Employee_5|   Finance|118603.0| 43|
+------+----------+----------+--------+---+
only showing top 5 rows



In [34]:
# 컬럼 목록 리스트 ["name", "department"]를 사용하여 해당 컬럼들을 선택
df2 = ["name","department"]
df.select(*df2).show(5)


+----------+----------+
|      name|department|
+----------+----------+
|Employee_1|        HR|
|Employee_2|   Finance|
|Employee_3| Marketing|
|Employee_4|   Finance|
|Employee_5|   Finance|
+----------+----------+
only showing top 5 rows



### 컬럼 추가/수정 (withColumn)

In [36]:
# 새 컬럼 추가
df_with_monthly = df.withColumn(
    "monthly_salary",
    col("salary") / 12
)

print("=== 월급 컬럼 추가 ===")
df_with_monthly.select("name","salary","monthly_salary").show(5)

=== 월급 컬럼 추가 ===
+----------+--------+------------------+
|      name|  salary|    monthly_salary|
+----------+--------+------------------+
|Employee_1| 92251.0| 7687.583333333333|
|Employee_2| 62662.0| 5221.833333333333|
|Employee_3| 48392.0|4032.6666666666665|
|Employee_4| 70535.0| 5877.916666666667|
|Employee_5|118603.0| 9883.583333333334|
+----------+--------+------------------+
only showing top 5 rows



In [37]:
df_with_monthly.show(5)

+------+----------+----------+--------+---+----------+------------------+
|emp_id|      name|department|  salary|age|is_manager|    monthly_salary|
+------+----------+----------+--------+---+----------+------------------+
|  E001|Employee_1|        HR| 92251.0| 28|     false| 7687.583333333333|
|  E002|Employee_2|   Finance| 62662.0| 43|     false| 5221.833333333333|
|  E003|Employee_3| Marketing| 48392.0| 50|     false|4032.6666666666665|
|  E004|Employee_4|   Finance| 70535.0| 27|     false| 5877.916666666667|
|  E005|Employee_5|   Finance|118603.0| 43|     false| 9883.583333333334|
+------+----------+----------+--------+---+----------+------------------+
only showing top 5 rows



In [38]:
# 상수 컬럼 추가
df_with_country = df.withColumn(
    "country",
    lit("Korea")
)
print("=== 상수 컬럼 추가 ===")
df_with_country.select("name","country").show(5)

=== 상수 컬럼 추가 ===
+----------+-------+
|      name|country|
+----------+-------+
|Employee_1|  Korea|
|Employee_2|  Korea|
|Employee_3|  Korea|
|Employee_4|  Korea|
|Employee_5|  Korea|
+----------+-------+
only showing top 5 rows



In [40]:
# 기존 컬럼 수정(덮어쓰기)
df_raised = df.withColumn(
    "salary",
    col("salary") * 1.1  # 10% 인상
)

print("=== 급여 10% 인상 (원본비교) ===")
print("원본")
df.select("name","salary").show(3)

print("=== 수정 후 ===")
df_raised.select("name","salary").show(3)

=== 급여 10% 인상 (원본비교) ===
원본
+----------+-------+
|      name| salary|
+----------+-------+
|Employee_1|92251.0|
|Employee_2|62662.0|
|Employee_3|48392.0|
+----------+-------+
only showing top 3 rows

=== 수정 후 ===
+----------+------------------+
|      name|            salary|
+----------+------------------+
|Employee_1|          101476.1|
|Employee_2| 68928.20000000001|
|Employee_3|53231.200000000004|
+----------+------------------+
only showing top 3 rows



In [42]:
# 체이닝(여러 컬럼 한번에)
# 여러 withColumn을 연속으로 호출(메서드 체이닝)

df_enhanced = (
    df
    # 연봉에서 월급 계산
    .withColumn("monthly_salary",col("salary")/12)
    # 연봉에서 일급 계산(연 250일 근부 가정)
    .withColumn("daily_salary", col("salary")/250)
    # 국가 추가
    .withColumn("country", lit("Korea"))
    # 연도 추가
    .withColumn("year", lit(2024))
)

print("=== 여러 컬럼 추가 ===")
df_enhanced.show(5)

=== 여러 컬럼 추가 ===
+------+----------+----------+--------+---+----------+------------------+------------+-------+----+
|emp_id|      name|department|  salary|age|is_manager|    monthly_salary|daily_salary|country|year|
+------+----------+----------+--------+---+----------+------------------+------------+-------+----+
|  E001|Employee_1|        HR| 92251.0| 28|     false| 7687.583333333333|     369.004|  Korea|2024|
|  E002|Employee_2|   Finance| 62662.0| 43|     false| 5221.833333333333|     250.648|  Korea|2024|
|  E003|Employee_3| Marketing| 48392.0| 50|     false|4032.6666666666665|     193.568|  Korea|2024|
|  E004|Employee_4|   Finance| 70535.0| 27|     false| 5877.916666666667|      282.14|  Korea|2024|
|  E005|Employee_5|   Finance|118603.0| 43|     false| 9883.583333333334|     474.412|  Korea|2024|
+------+----------+----------+--------+---+----------+------------------+------------+-------+----+
only showing top 5 rows



### 컬럼 이름 변경과 삭제

In [49]:
# withColumnRenamed() : 컬럼 이름 변경
df_renamed = df.withColumnRenamed("emp_id", "employee_id")

df_renamed.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- age: integer (nullable = true)
 |-- is_manager: boolean (nullable = true)



In [ ]:
# 여러 컬럼 이름 변경
# 방법1 : withColumnRenamed 체이닝
df_multi_renamed = (
    df
    .withColumnRenamed("emp_id", "employee_id")
    .withColumnRenamed("department", "dept")
    .withColumnRenamed("is_manager", "manager_flag")
)

df_multi_renamed.printSchema()

# 방법 2: toDF() - 모든 컬럼명 한번에 변경
# 주의: 컬럼 순서와 개수가 정확히 일치해야 함
# df.toDF("new_col1", "new_col2", ...) - 모든 컬럼에 새 이름 지정

root
 |-- employee_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- age: integer (nullable = true)
 |-- manager_flag: boolean (nullable = true)



In [51]:
# 단일 컬럼 삭제
df_no_manager = df.drop("is_manager")

print(f"삭제 전 컬럼: {df.columns}")
print(f"삭제 후 컬럼: {df_no_manager.columns}")

삭제 전 컬럼: ['emp_id', 'name', 'department', 'salary', 'age', 'is_manager']
삭제 후 컬럼: ['emp_id', 'name', 'department', 'salary', 'age']


In [52]:
# 여러 컬럼 삭제
df_minimal = df.drop("is_manager", "age")
print(f"여러 컬럼 삭제 후: {df_minimal.columns}")

여러 컬럼 삭제 후: ['emp_id', 'name', 'department', 'salary']


### 실습 3, 4

In [53]:
# age에 1을 더한 "next_age" 컬럼을 추가
df.withColumn("next_age", col("age") + 1).show(5)

+------+----------+----------+--------+---+----------+--------+
|emp_id|      name|department|  salary|age|is_manager|next_age|
+------+----------+----------+--------+---+----------+--------+
|  E001|Employee_1|        HR| 92251.0| 28|     false|      29|
|  E002|Employee_2|   Finance| 62662.0| 43|     false|      44|
|  E003|Employee_3| Marketing| 48392.0| 50|     false|      51|
|  E004|Employee_4|   Finance| 70535.0| 27|     false|      28|
|  E005|Employee_5|   Finance|118603.0| 43|     false|      44|
+------+----------+----------+--------+---+----------+--------+
only showing top 5 rows



In [54]:
# 모든 행에 "2024"라는 값을 가진 "year" 컬럼을 추가
df.withColumn("year", lit("2024")).show(5)

+------+----------+----------+--------+---+----------+----+
|emp_id|      name|department|  salary|age|is_manager|year|
+------+----------+----------+--------+---+----------+----+
|  E001|Employee_1|        HR| 92251.0| 28|     false|2024|
|  E002|Employee_2|   Finance| 62662.0| 43|     false|2024|
|  E003|Employee_3| Marketing| 48392.0| 50|     false|2024|
|  E004|Employee_4|   Finance| 70535.0| 27|     false|2024|
|  E005|Employee_5|   Finance|118603.0| 43|     false|2024|
+------+----------+----------+--------+---+----------+----+
only showing top 5 rows



In [55]:
# age 컬럼의 값을 모두 10 증가시킨 DataFrame을 만들기
df_plus_age = df.withColumn("age",col("age")+10)
df_plus_age.show(5)

+------+----------+----------+--------+---+----------+
|emp_id|      name|department|  salary|age|is_manager|
+------+----------+----------+--------+---+----------+
|  E001|Employee_1|        HR| 92251.0| 38|     false|
|  E002|Employee_2|   Finance| 62662.0| 53|     false|
|  E003|Employee_3| Marketing| 48392.0| 60|     false|
|  E004|Employee_4|   Finance| 70535.0| 37|     false|
|  E005|Employee_5|   Finance|118603.0| 53|     false|
+------+----------+----------+--------+---+----------+
only showing top 5 rows



In [57]:
# salary의 20%를 "tax", salary에서 tax를 뺀 "net_salary" 컬럼을 추가
df_tax = (
    df
    .withColumn("tax",col("salary")*0.2)
    .withColumn("net_salary", col("salary")-col("tax"))
)

df_tax.show(5)

+------+----------+----------+--------+---+----------+------------------+----------+
|emp_id|      name|department|  salary|age|is_manager|               tax|net_salary|
+------+----------+----------+--------+---+----------+------------------+----------+
|  E001|Employee_1|        HR| 92251.0| 28|     false|           18450.2|   73800.8|
|  E002|Employee_2|   Finance| 62662.0| 43|     false|12532.400000000001|   50129.6|
|  E003|Employee_3| Marketing| 48392.0| 50|     false|            9678.4|   38713.6|
|  E004|Employee_4|   Finance| 70535.0| 27|     false|           14107.0|   56428.0|
|  E005|Employee_5|   Finance|118603.0| 43|     false|23720.600000000002|   94882.4|
+------+----------+----------+--------+---+----------+------------------+----------+
only showing top 5 rows



In [58]:
# salary 컬럼의 이름을 "annual_salary"로 변경
df.withColumnRenamed("salary","annual_salary").printSchema()

root
 |-- emp_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- annual_salary: double (nullable = true)
 |-- age: integer (nullable = true)
 |-- is_manager: boolean (nullable = true)



In [59]:
# emp_id를 "id"로, name을 "employee_name"으로 변경
df_rename = (
    df
    .withColumnRenamed("emp_id","id")
    .withColumnRenamed("name","employee_name")
)

df_rename.printSchema()

root
 |-- id: string (nullable = true)
 |-- employee_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- age: integer (nullable = true)
 |-- is_manager: boolean (nullable = true)



In [61]:
# age와 is_manager 두 컬럼을 삭제
df_drop = df.drop("age","is_manager")
df_drop.columns

['emp_id', 'name', 'department', 'salary']

### 필터링(filter / where)

In [65]:
# 기본 필터링
high_salary = df.filter(col("salary") >= 80000)
high_salary.show(5)

engineers = df.filter(col("department") == "Engineering")
engineers.show(5)

+------+-----------+----------+--------+---+----------+
|emp_id|       name|department|  salary|age|is_manager|
+------+-----------+----------+--------+---+----------+
|  E001| Employee_1|        HR| 92251.0| 28|     false|
|  E005| Employee_5|   Finance|118603.0| 43|     false|
|  E013|Employee_13|   Finance|110592.0| 42|     false|
|  E015|Employee_15|        HR|119309.0| 25|      true|
|  E017|Employee_17|      NULL| 92992.0| 52|     false|
+------+-----------+----------+--------+---+----------+
only showing top 5 rows

+------+-----------+-----------+--------+---+----------+
|emp_id|       name| department|  salary|age|is_manager|
+------+-----------+-----------+--------+---+----------+
|  E024|Employee_24|Engineering| 58141.0| 27|     false|
|  E025|Employee_25|Engineering|111910.0| 31|     false|
|  E034|Employee_34|Engineering|104044.0| 26|     false|
|  E039|Employee_39|Engineering| 65939.0| 29|     false|
|  E042|Employee_42|Engineering| 61834.0| 47|     false|
+------+-------

In [68]:
# AND / OR 조건
senior_high = df.filter((col("age") >= 30) & (col("salary") >= 70000))
senior_high.show(5)

eng_or_sales = df.filter((col("department") == "Engineering") | (col("department") == "Sales"))
eng_or_sales.show(5)

+------+-----------+----------+--------+---+----------+
|emp_id|       name|department|  salary|age|is_manager|
+------+-----------+----------+--------+---+----------+
|  E005| Employee_5|   Finance|118603.0| 43|     false|
|  E013|Employee_13|   Finance|110592.0| 42|     false|
|  E017|Employee_17|      NULL| 92992.0| 52|     false|
|  E021|Employee_21|     Sales| 90636.0| 47|     false|
|  E022|Employee_22|   Finance| 90015.0| 54|     false|
+------+-----------+----------+--------+---+----------+
only showing top 5 rows

+------+-----------+-----------+--------+---+----------+
|emp_id|       name| department|  salary|age|is_manager|
+------+-----------+-----------+--------+---+----------+
|  E006| Employee_6|      Sales|    NULL| 44|     false|
|  E014|Employee_14|      Sales| 48110.0| 32|     false|
|  E021|Employee_21|      Sales| 90636.0| 47|     false|
|  E024|Employee_24|Engineering| 58141.0| 27|     false|
|  E025|Employee_25|Engineering|111910.0| 31|     false|
+------+-------

In [69]:
# -----------------------------------------------------------------------------
# filter(): SQL 스타일 문자열 조건
# -----------------------------------------------------------------------------

# 문자열로 SQL WHERE 절처럼 작성 가능
# 더 직관적일 수 있음 (SQL에 익숙하면)

# SQL 스타일 필터링
df.filter("age >= 30 AND salary >= 70000").show(5)

# SQL 스타일: BETWEEN
df.filter("salary BETWEEN 50000 AND 80000").show(5)

# SQL 스타일: IN
df.filter("department IN ('Engineering', 'Sales', 'Marketing')").show(5)

# SQL 스타일: LIKE (문자열 패턴)
df.filter("name LIKE 'Employee_1%'").show(5)

+------+-----------+----------+--------+---+----------+
|emp_id|       name|department|  salary|age|is_manager|
+------+-----------+----------+--------+---+----------+
|  E005| Employee_5|   Finance|118603.0| 43|     false|
|  E013|Employee_13|   Finance|110592.0| 42|     false|
|  E017|Employee_17|      NULL| 92992.0| 52|     false|
|  E021|Employee_21|     Sales| 90636.0| 47|     false|
|  E022|Employee_22|   Finance| 90015.0| 54|     false|
+------+-----------+----------+--------+---+----------+
only showing top 5 rows

+------+-----------+----------+-------+---+----------+
|emp_id|       name|department| salary|age|is_manager|
+------+-----------+----------+-------+---+----------+
|  E002| Employee_2|   Finance|62662.0| 43|     false|
|  E004| Employee_4|   Finance|70535.0| 27|     false|
|  E012|Employee_12| Marketing|64538.0| 31|     false|
|  E016|Employee_16|      NULL|67266.0| 35|     false|
|  E020|Employee_20|        HR|63419.0| 42|     false|
+------+-----------+----------+

In [70]:
# isin(): 여러 값 중 하나인지 확인
# SQL의 IN 절과 동일
target_depts = ["Engineering", "Sales"]
df.filter(col("department").isin(target_depts)).show(5)

+------+-----------+-----------+--------+---+----------+
|emp_id|       name| department|  salary|age|is_manager|
+------+-----------+-----------+--------+---+----------+
|  E006| Employee_6|      Sales|    NULL| 44|     false|
|  E014|Employee_14|      Sales| 48110.0| 32|     false|
|  E021|Employee_21|      Sales| 90636.0| 47|     false|
|  E024|Employee_24|Engineering| 58141.0| 27|     false|
|  E025|Employee_25|Engineering|111910.0| 31|     false|
+------+-----------+-----------+--------+---+----------+
only showing top 5 rows



In [71]:
# isin()에 리스트 직접 전달
df.filter(col("department").isin("HR", "Finance")).show(5)

+------+-----------+----------+--------+---+----------+
|emp_id|       name|department|  salary|age|is_manager|
+------+-----------+----------+--------+---+----------+
|  E001| Employee_1|        HR| 92251.0| 28|     false|
|  E002| Employee_2|   Finance| 62662.0| 43|     false|
|  E004| Employee_4|   Finance| 70535.0| 27|     false|
|  E005| Employee_5|   Finance|118603.0| 43|     false|
|  E010|Employee_10|   Finance|    NULL| 25|     false|
+------+-----------+----------+--------+---+----------+
only showing top 5 rows



In [73]:
# isNull(): NULL인 행만
df.filter(col("salary").isNull()).show()

# isNotNull(): NULL이 아닌 행만
df.filter(col("salary").isNotNull()).count()

+------+-----------+----------+------+---+----------+
|emp_id|       name|department|salary|age|is_manager|
+------+-----------+----------+------+---+----------+
|  E006| Employee_6|     Sales|  NULL| 44|     false|
|  E007| Employee_7| Marketing|  NULL| 31|     false|
|  E008| Employee_8| Marketing|  NULL| 44|     false|
|  E009| Employee_9| Marketing|  NULL| 33|     false|
|  E010|Employee_10|   Finance|  NULL| 25|     false|
|  E011|Employee_11|        HR|  NULL| 32|     false|
+------+-----------+----------+------+---+----------+



94

In [74]:
# -----------------------------------------------------------------------------
# filter(): 문자열 조건 메서드
# -----------------------------------------------------------------------------

# startswith(): 특정 문자로 시작
df.filter(col("name").startswith("Employee_1")).show(5)

# endswith(): 특정 문자로 끝
df.filter(col("emp_id").endswith("5")).show(5)

# contains(): 특정 문자 포함
df.filter(col("department").contains("ing")).show(5)  # Engineering, Marketing

+------+-----------+----------+--------+---+----------+
|emp_id|       name|department|  salary|age|is_manager|
+------+-----------+----------+--------+---+----------+
|  E001| Employee_1|        HR| 92251.0| 28|     false|
|  E010|Employee_10|   Finance|    NULL| 25|     false|
|  E011|Employee_11|        HR|    NULL| 32|     false|
|  E012|Employee_12| Marketing| 64538.0| 31|     false|
|  E013|Employee_13|   Finance|110592.0| 42|     false|
+------+-----------+----------+--------+---+----------+
only showing top 5 rows

+------+-----------+-----------+--------+---+----------+
|emp_id|       name| department|  salary|age|is_manager|
+------+-----------+-----------+--------+---+----------+
|  E005| Employee_5|    Finance|118603.0| 43|     false|
|  E015|Employee_15|         HR|119309.0| 25|      true|
|  E025|Employee_25|Engineering|111910.0| 31|     false|
|  E035|Employee_35|  Marketing| 82557.0| 25|      true|
|  E045|Employee_45|      Sales|115766.0| 27|     false|
+------+-------

In [75]:
# -----------------------------------------------------------------------------
# where(): filter()와 동일 (SQL 친화적 이름)
# -----------------------------------------------------------------------------

# where()는 filter()의 별칭 (alias)
# SQL에 익숙한 사람을 위한 이름

# filter()와 완전히 동일한 동작
df.where(col("age") >= 30).show(5)
df.where("salary > 60000").show(5)

+------+----------+----------+--------+---+----------+
|emp_id|      name|department|  salary|age|is_manager|
+------+----------+----------+--------+---+----------+
|  E002|Employee_2|   Finance| 62662.0| 43|     false|
|  E003|Employee_3| Marketing| 48392.0| 50|     false|
|  E005|Employee_5|   Finance|118603.0| 43|     false|
|  E006|Employee_6|     Sales|    NULL| 44|     false|
|  E007|Employee_7| Marketing|    NULL| 31|     false|
+------+----------+----------+--------+---+----------+
only showing top 5 rows

+------+-----------+----------+--------+---+----------+
|emp_id|       name|department|  salary|age|is_manager|
+------+-----------+----------+--------+---+----------+
|  E001| Employee_1|        HR| 92251.0| 28|     false|
|  E002| Employee_2|   Finance| 62662.0| 43|     false|
|  E004| Employee_4|   Finance| 70535.0| 27|     false|
|  E005| Employee_5|   Finance|118603.0| 43|     false|
|  E012|Employee_12| Marketing| 64538.0| 31|     false|
+------+-----------+----------+-

### 실습 5

In [78]:
# salary가 70000 이상인 직원만 조회
df.filter("salary >= 70000").show(5)

+------+-----------+----------+--------+---+----------+
|emp_id|       name|department|  salary|age|is_manager|
+------+-----------+----------+--------+---+----------+
|  E001| Employee_1|        HR| 92251.0| 28|     false|
|  E004| Employee_4|   Finance| 70535.0| 27|     false|
|  E005| Employee_5|   Finance|118603.0| 43|     false|
|  E013|Employee_13|   Finance|110592.0| 42|     false|
|  E015|Employee_15|        HR|119309.0| 25|      true|
+------+-----------+----------+--------+---+----------+
only showing top 5 rows



In [82]:
# department가 "Engineering"이면서 age가 35 이상인 직원을 조회
df.filter(
    (col("department") == "Engineering") & (col("Age") >= 35)
).show(5)

+------+-----------+-----------+-------+---+----------+
|emp_id|       name| department| salary|age|is_manager|
+------+-----------+-----------+-------+---+----------+
|  E042|Employee_42|Engineering|61834.0| 47|     false|
|  E046|Employee_46|Engineering|55707.0| 43|     false|
|  E056|Employee_56|Engineering|98053.0| 44|      true|
|  E071|Employee_71|Engineering|92733.0| 53|     false|
|  E085|Employee_85|Engineering|92662.0| 38|     false|
+------+-----------+-----------+-------+---+----------+
only showing top 5 rows



In [83]:
# department가 "Sales" 또는 "Marketing"인 직원을 조회
df.filter("department IN ('Sales', 'Marketing')").show(5)

+------+----------+----------+-------+---+----------+
|emp_id|      name|department| salary|age|is_manager|
+------+----------+----------+-------+---+----------+
|  E003|Employee_3| Marketing|48392.0| 50|     false|
|  E006|Employee_6|     Sales|   NULL| 44|     false|
|  E007|Employee_7| Marketing|   NULL| 31|     false|
|  E008|Employee_8| Marketing|   NULL| 44|     false|
|  E009|Employee_9| Marketing|   NULL| 33|     false|
+------+----------+----------+-------+---+----------+
only showing top 5 rows



In [86]:
#salary가 NULL인 행을 조회
df.filter(col("salary").isNull()).show(5)

+------+-----------+----------+------+---+----------+
|emp_id|       name|department|salary|age|is_manager|
+------+-----------+----------+------+---+----------+
|  E006| Employee_6|     Sales|  NULL| 44|     false|
|  E007| Employee_7| Marketing|  NULL| 31|     false|
|  E008| Employee_8| Marketing|  NULL| 44|     false|
|  E009| Employee_9| Marketing|  NULL| 33|     false|
|  E010|Employee_10|   Finance|  NULL| 25|     false|
+------+-----------+----------+------+---+----------+
only showing top 5 rows



In [87]:
# name이 "Employee_1"로 시작하는 직원을 조회
df.filter(col("name").startswith("Employee_1")).show(5)

+------+-----------+----------+--------+---+----------+
|emp_id|       name|department|  salary|age|is_manager|
+------+-----------+----------+--------+---+----------+
|  E001| Employee_1|        HR| 92251.0| 28|     false|
|  E010|Employee_10|   Finance|    NULL| 25|     false|
|  E011|Employee_11|        HR|    NULL| 32|     false|
|  E012|Employee_12| Marketing| 64538.0| 31|     false|
|  E013|Employee_13|   Finance|110592.0| 42|     false|
+------+-----------+----------+--------+---+----------+
only showing top 5 rows



### 조건부 값 설정 (When/otherwise)

In [88]:
# 단일 조건
df_adult = df.withColumn(
    "is_adult",
    when(col("age") >= 18, "Yes").otherwise("No")
)
df_adult.select("name","age","is_adult").show(5)

+----------+---+--------+
|      name|age|is_adult|
+----------+---+--------+
|Employee_1| 28|     Yes|
|Employee_2| 43|     Yes|
|Employee_3| 50|     Yes|
|Employee_4| 27|     Yes|
|Employee_5| 43|     Yes|
+----------+---+--------+
only showing top 5 rows



In [91]:
# 다중 조건
df_age_group = df.withColumn(
    "age_group",
    when(col("age") >= 50, "50대 이상")
    .when(col("age") >= 40, "40대")
    .when(col("age") >= 30, "30대")
    .otherwise("20대")
)
df_age_group.select("name","age","age_group").show(10)

df_age_group.groupBy("age_group").count().show()

+-----------+---+---------+
|       name|age|age_group|
+-----------+---+---------+
| Employee_1| 28|     20대|
| Employee_2| 43|     40대|
| Employee_3| 50|50대 이상|
| Employee_4| 27|     20대|
| Employee_5| 43|     40대|
| Employee_6| 44|     40대|
| Employee_7| 31|     30대|
| Employee_8| 44|     40대|
| Employee_9| 33|     30대|
|Employee_10| 25|     20대|
+-----------+---+---------+
only showing top 10 rows

+---------+-----+
|age_group|count|
+---------+-----+
|     20대|   23|
|50대 이상|   13|
|     40대|   43|
|     30대|   21|
+---------+-----+



### 실습 6

In [95]:
# is_manager가 True이면 "Manager", 아니면 "Staff"인 "role" 컬럼을 추가
df.withColumn(
    "role",
    when(col("is_manager") == "True", "Manager").otherwise("Staff")
).show(20)

+------+-----------+----------+--------+---+----------+-------+
|emp_id|       name|department|  salary|age|is_manager|   role|
+------+-----------+----------+--------+---+----------+-------+
|  E001| Employee_1|        HR| 92251.0| 28|     false|  Staff|
|  E002| Employee_2|   Finance| 62662.0| 43|     false|  Staff|
|  E003| Employee_3| Marketing| 48392.0| 50|     false|  Staff|
|  E004| Employee_4|   Finance| 70535.0| 27|     false|  Staff|
|  E005| Employee_5|   Finance|118603.0| 43|     false|  Staff|
|  E006| Employee_6|     Sales|    NULL| 44|     false|  Staff|
|  E007| Employee_7| Marketing|    NULL| 31|     false|  Staff|
|  E008| Employee_8| Marketing|    NULL| 44|     false|  Staff|
|  E009| Employee_9| Marketing|    NULL| 33|     false|  Staff|
|  E010|Employee_10|   Finance|    NULL| 25|     false|  Staff|
|  E011|Employee_11|        HR|    NULL| 32|     false|  Staff|
|  E012|Employee_12| Marketing| 64538.0| 31|     false|  Staff|
|  E013|Employee_13|   Finance|110592.0|

In [96]:
# age 기준으로 "age_band" 컬럼을 추가
df.withColumn(
    "age_band",
    when(col("age") >= 40, "40+")
    .when(col("age") >= 30, "30s")
    .otherwise("20s")
).show(20)

+------+-----------+----------+--------+---+----------+--------+
|emp_id|       name|department|  salary|age|is_manager|age_band|
+------+-----------+----------+--------+---+----------+--------+
|  E001| Employee_1|        HR| 92251.0| 28|     false|     20s|
|  E002| Employee_2|   Finance| 62662.0| 43|     false|     40+|
|  E003| Employee_3| Marketing| 48392.0| 50|     false|     40+|
|  E004| Employee_4|   Finance| 70535.0| 27|     false|     20s|
|  E005| Employee_5|   Finance|118603.0| 43|     false|     40+|
|  E006| Employee_6|     Sales|    NULL| 44|     false|     40+|
|  E007| Employee_7| Marketing|    NULL| 31|     false|     30s|
|  E008| Employee_8| Marketing|    NULL| 44|     false|     40+|
|  E009| Employee_9| Marketing|    NULL| 33|     false|     30s|
|  E010|Employee_10|   Finance|    NULL| 25|     false|     20s|
|  E011|Employee_11|        HR|    NULL| 32|     false|     30s|
|  E012|Employee_12| Marketing| 64538.0| 31|     false|     30s|
|  E013|Employee_13|   Fi

In [97]:
# salary 기준으로 "salary_level" 컬럼을 추가
df.withColumn(
    "salary_level",
    when(col("salary") >= 90000, "High")
    .when(col("salary") >= 60000, "Medium")
    .otherwise("Low")
).show(10)

+------+-----------+----------+--------+---+----------+------------+
|emp_id|       name|department|  salary|age|is_manager|salary_level|
+------+-----------+----------+--------+---+----------+------------+
|  E001| Employee_1|        HR| 92251.0| 28|     false|        High|
|  E002| Employee_2|   Finance| 62662.0| 43|     false|      Medium|
|  E003| Employee_3| Marketing| 48392.0| 50|     false|         Low|
|  E004| Employee_4|   Finance| 70535.0| 27|     false|      Medium|
|  E005| Employee_5|   Finance|118603.0| 43|     false|        High|
|  E006| Employee_6|     Sales|    NULL| 44|     false|         Low|
|  E007| Employee_7| Marketing|    NULL| 31|     false|         Low|
|  E008| Employee_8| Marketing|    NULL| 44|     false|         Low|
|  E009| Employee_9| Marketing|    NULL| 33|     false|         Low|
|  E010|Employee_10|   Finance|    NULL| 25|     false|         Low|
+------+-----------+----------+--------+---+----------+------------+
only showing top 10 rows



### 정렬, 중복 제거, 제한

In [99]:
# 오름차순
df.orderBy(col("salary")).select("name","salary").show(5)

+-----------+------+
|       name|salary|
+-----------+------+
| Employee_7|  NULL|
| Employee_8|  NULL|
| Employee_9|  NULL|
| Employee_6|  NULL|
|Employee_11|  NULL|
+-----------+------+
only showing top 5 rows



In [102]:
# 내림차순
df.orderBy(col("salary").desc()).select("name","salary").show(5)

+-----------+--------+
|       name|  salary|
+-----------+--------+
|Employee_15|119309.0|
| Employee_5|118603.0|
|Employee_45|115766.0|
|Employee_31|115450.0|
|Employee_63|113530.0|
+-----------+--------+
only showing top 5 rows



In [103]:
# 여러 컬럼 정렬 : 부서(오름차순), 급여(내림차순)
df.orderBy(
    col("department").asc(),
    col("salary").desc()
).select("department", "name", "salary").show(10)

+-----------+------------+--------+
| department|        name|  salary|
+-----------+------------+--------+
|       NULL| Employee_17| 92992.0|
|       NULL| Employee_16| 67266.0|
|       NULL| Employee_18| 46910.0|
|       NULL| Employee_19| 40206.0|
|Engineering| Employee_25|111910.0|
|Engineering| Employee_34|104044.0|
|Engineering| Employee_56| 98053.0|
|Engineering|Employee_100| 96250.0|
|Engineering| Employee_71| 92733.0|
|Engineering| Employee_85| 92662.0|
+-----------+------------+--------+
only showing top 10 rows



In [104]:
# 중복 제거
# distinct() : 전체 행이 동일한 중복 제거

departments = df.select("department").distinct()
departments.show()

+-----------+
| department|
+-----------+
|         HR|
|  Marketing|
|      Sales|
|Engineering|
|    Finance|
|       NULL|
+-----------+



In [109]:
# dropDuplicates() : 특정 컬럼 기준 중복 제거
df.dropDuplicates(["department","is_manager"]).select("department","name","is_manager").show()

+-----------+-----------+----------+
| department|       name|is_manager|
+-----------+-----------+----------+
|       NULL|Employee_16|     false|
|Engineering|Employee_24|     false|
|Engineering|Employee_56|      true|
|    Finance| Employee_2|     false|
|    Finance|Employee_54|      true|
|         HR| Employee_1|     false|
|         HR|Employee_15|      true|
|  Marketing| Employee_3|     false|
|  Marketing|Employee_26|      true|
|      Sales| Employee_6|     false|
|      Sales|Employee_28|      true|
+-----------+-----------+----------+



### 실습 7

In [ ]:
# salary 기준 내림차순으로 정렬하여 상위 10명을 조회
df.orderBy(col("salary").desc()).limit(10).select("name","salary").show()

+-----------+--------+
|       name|  salary|
+-----------+--------+
|Employee_15|119309.0|
| Employee_5|118603.0|
|Employee_45|115766.0|
|Employee_31|115450.0|
|Employee_63|113530.0|
|Employee_25|111910.0|
|Employee_13|110592.0|
|Employee_84|110467.0|
|Employee_38|109163.0|
|Employee_65|108840.0|
+-----------+--------+



In [ ]:
# department 오름차순, age 내림차순으로 정렬
df.orderBy(col("department").asc(), col("age").desc()).select("department","name","age").show(10)

+-----------+------------+---+
| department|        name|age|
+-----------+------------+---+
|       NULL| Employee_17| 52|
|       NULL| Employee_18| 49|
|       NULL| Employee_19| 49|
|       NULL| Employee_16| 35|
|Engineering| Employee_71| 53|
|Engineering| Employee_42| 47|
|Engineering|Employee_100| 47|
|Engineering| Employee_86| 45|
|Engineering| Employee_56| 44|
|Engineering| Employee_88| 44|
+-----------+------------+---+
only showing top 10 rows



In [ ]:
# 고유한 department 목록을 조회
df.select("department").distinct().show()

+-----------+
| department|
+-----------+
|         HR|
|  Marketing|
|      Sales|
|Engineering|
|    Finance|
|       NULL|
+-----------+



In [ ]:
# 급여가 가장 높은 직원 3명의 이름과 급여를 조회
df.orderBy(col("salary").desc()).limit(3).select("name","salary").show()


+-----------+--------+
|       name|  salary|
+-----------+--------+
|Employee_15|119309.0|
| Employee_5|118603.0|
|Employee_45|115766.0|
+-----------+--------+



### 과제 : 인사팀 월간 리포트 생성

In [131]:
# Step 1: 컬럼 선택 및 이름 정리
df_step1 = df.select("emp_id","name","department","salary","age")

df_step1 = (
    df
    .withColumnRenamed("emp_id","사원번호")
    .withColumnRenamed("name","이름")
    .withColumnRenamed("department","부서")
    .withColumnRenamed("salary","연봉")
    .withColumnRenamed("age","나이")
)

df_step1.show(5)


+--------+----------+---------+--------+----+----------+
|사원번호|      이름|     부서|    연봉|나이|is_manager|
+--------+----------+---------+--------+----+----------+
|    E001|Employee_1|       HR| 92251.0|  28|     false|
|    E002|Employee_2|  Finance| 62662.0|  43|     false|
|    E003|Employee_3|Marketing| 48392.0|  50|     false|
|    E004|Employee_4|  Finance| 70535.0|  27|     false|
|    E005|Employee_5|  Finance|118603.0|  43|     false|
+--------+----------+---------+--------+----+----------+
only showing top 5 rows



In [ ]:
# Step 2: 파생 컬럼 추가
df_step2 = (
    df_step1.withColumn(
        "급여등급",
        when(col("연봉") >= 100000 , "S")
        .when(col("연봉") >= 80000, "A")
        .when(col("연봉") >= 60000, "B")
        .otherwise("C")
    )
    .withColumn("세후연봉", (col("연봉")*0.8).cast("int"))
    .withColumn("월급", (col("연봉")/12).cast("int"))
)

df_step2.show(5)

+--------+----------+---------+--------+----+----------+--------+--------+----+
|사원번호|      이름|     부서|    연봉|나이|is_manager|급여등급|세후연봉|월급|
+--------+----------+---------+--------+----+----------+--------+--------+----+
|    E001|Employee_1|       HR| 92251.0|  28|     false|       A|   73800|7687|
|    E002|Employee_2|  Finance| 62662.0|  43|     false|       B|   50129|5221|
|    E003|Employee_3|Marketing| 48392.0|  50|     false|       C|   38713|4032|
|    E004|Employee_4|  Finance| 70535.0|  27|     false|       B|   56428|5877|
|    E005|Employee_5|  Finance|118603.0|  43|     false|       S|   94882|9883|
+--------+----------+---------+--------+----+----------+--------+--------+----+
only showing top 5 rows



In [127]:
# Step 3: 고성과자 필터링
df_step3 = df_step2.filter(
    (col("급여등급").isin("S","A")) &
    (col("나이") >= 35) &
    (col("부서").isNotNull())
)

df_step3.show(5)
print(f"고 성과자 수: {df_step3.count()}명")

+--------+-----------+---------+--------+----+----------+--------+--------+----+
|사원번호|       이름|     부서|    연봉|나이|is_manager|급여등급|세후연봉|월급|
+--------+-----------+---------+--------+----+----------+--------+--------+----+
|    E005| Employee_5|  Finance|118603.0|  43|     false|       S|   94882|9883|
|    E013|Employee_13|  Finance|110592.0|  42|     false|       S|   88473|9216|
|    E021|Employee_21|    Sales| 90636.0|  47|     false|       A|   72508|7553|
|    E022|Employee_22|  Finance| 90015.0|  54|     false|       A|   72012|7501|
|    E026|Employee_26|Marketing| 96044.0|  52|      true|       A|   76835|8003|
+--------+-----------+---------+--------+----+----------+--------+--------+----+
only showing top 5 rows

고 성과자 수: 33명


In [130]:
# Step 4: 정렬 및 최종 정리
df_step4 = (
    df_step3
    .orderBy(col("연봉").asc(), col("이름").desc())
    .limit(20)
    .drop("나이")
)

df_step4.show()

+--------+------------+-----------+--------+----------+--------+--------+----+
|사원번호|        이름|       부서|    연봉|is_manager|급여등급|세후연봉|월급|
+--------+------------+-----------+--------+----------+--------+--------+----+
|    E095| Employee_95|Engineering| 82107.0|     false|       A|   65685|6842|
|    E041| Employee_41|         HR| 82941.0|     false|       A|   66352|6911|
|    E033| Employee_33|         HR| 83585.0|     false|       A|   66868|6965|
|    E048| Employee_48|    Finance| 84262.0|     false|       A|   67409|7021|
|    E068| Employee_68|      Sales| 86576.0|      true|       A|   69260|7214|
|    E036| Employee_36|    Finance| 89080.0|     false|       A|   71264|7423|
|    E098| Employee_98|      Sales| 89811.0|      true|       A|   71848|7484|
|    E022| Employee_22|    Finance| 90015.0|     false|       A|   72012|7501|
|    E021| Employee_21|      Sales| 90636.0|     false|       A|   72508|7553|
|    E067| Employee_67|      Sales| 91005.0|     false|       A|   72804

### 2교시: 집계와 조인

In [ ]:
# -----------------------------------------------------------------------------
# 환경 설정: SparkSession 및 테스트 데이터 준비
# -----------------------------------------------------------------------------
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, when, count, sum, avg, min, max,
    countDistinct, first, last, collect_list, collect_set,
    round as spark_round, expr
)
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import pandas as pd
import numpy as np
import os

# SparkSession 생성
spark = SparkSession.builder \
    .appName("PySpark-Aggregation-Join") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", 10) \
    .getOrCreate()

# 테스트 데이터 디렉토리
os.makedirs("/tmp/spark_tutorial", exist_ok=True)

np.random.seed(42)

# -----------------------------------------------------------------------------
# 테스트 데이터 1: 직원 정보
# -----------------------------------------------------------------------------
employees = pd.DataFrame({
    "emp_id": [f"E{i:03d}" for i in range(1, 51)],
    "name": [f"Employee_{i}" for i in range(1, 51)],
    "department": np.random.choice(
        ["Engineering", "Sales", "Marketing", "HR", "Finance"], 50
    ),
    "salary": np.random.randint(40000, 120000, 50),
    "age": np.random.randint(25, 55, 50),
    "hire_year": np.random.choice([2020, 2021, 2022, 2023, 2024], 50),
})
employees.to_csv("/tmp/spark_tutorial/employees.csv", index=False)

# -----------------------------------------------------------------------------
# 테스트 데이터 2: 부서 정보 (조인용)
# -----------------------------------------------------------------------------
departments = pd.DataFrame({
    "dept_name": ["Engineering", "Sales", "Marketing", "HR", "Finance", "Legal"],
    "dept_head": ["Alice", "Bob", "Charlie", "Diana", "Eve", "Frank"],
    "budget": [500000, 300000, 200000, 150000, 400000, 100000],
    "location": ["Seoul", "Busan", "Seoul", "Daegu", "Seoul", "Incheon"],
})
departments.to_csv("/tmp/spark_tutorial/departments.csv", index=False)

# -----------------------------------------------------------------------------
# 테스트 데이터 3: 매출 데이터 (시계열)
# -----------------------------------------------------------------------------
sales = pd.DataFrame({
    "date": pd.date_range("2026-01-01", periods=100, freq="D").strftime("%Y-%m-%d"),
    "product": np.random.choice(["A", "B", "C"], 100),
    "region": np.random.choice(["Seoul", "Busan", "Daegu"], 100),
    "amount": np.random.randint(100, 1000, 100),
    "quantity": np.random.randint(1, 20, 100),
})
sales.to_csv("/tmp/spark_tutorial/sales.csv", index=False)

# DataFrame 로드
df_emp = spark.read.csv("/tmp/spark_tutorial/employees.csv", header=True, inferSchema=True)
df_dept = spark.read.csv("/tmp/spark_tutorial/departments.csv", header=True, inferSchema=True)
df_sales = spark.read.csv("/tmp/spark_tutorial/sales.csv", header=True, inferSchema=True)

print("데이터 로드 완료!")
print(f"직원: {df_emp.count()}명, 부서: {df_dept.count()}개, 매출: {df_sales.count()}건")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/20 06:31:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/20 06:31:25 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


데이터 로드 완료!
직원: 50명, 부서: 6개, 매출: 100건


26/01/20 06:31:40 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [133]:
# -----------------------------------------------------------------------------
# agg(): 전체 데이터 집계 (groupBy 없이)
# -----------------------------------------------------------------------------

# agg(): 집계 함수들을 적용하여 결과 DataFrame 반환
# 여러 집계를 한 번에 계산 가능

# 전체 직원 통계
total_stats = df_emp.agg(
    count("*").alias("총인원"),                    # 전체 행 수
    count("salary").alias("급여있는인원"),          # NULL 제외 카운트
    sum("salary").alias("총급여"),                 # 급여 합계
    avg("salary").alias("평균급여"),               # 급여 평균
    min("salary").alias("최소급여"),               # 최소 급여
    max("salary").alias("최대급여"),               # 최대 급여
    countDistinct("department").alias("부서수"),   # 고유 부서 수
)

print("=== 전체 직원 통계 ===")
total_stats.show()

=== 전체 직원 통계 ===
+------+------------+-------+--------+--------+--------+------+
|총인원|급여있는인원| 총급여|평균급여|최소급여|최대급여|부서수|
+------+------------+-------+--------+--------+--------+------+
|    50|          50|3954398|79087.96|   41016|  118953|     5|
+------+------------+-------+--------+--------+--------+------+



### 실습

In [135]:
# 직원 데이터의 총 행 수와 평균 급여를 한 번에 조회
df_emp.agg(
    count("*").alias("총인원"), 
    avg("salary").alias("평균급여")
).show()

+------+--------+
|총인원|평균급여|
+------+--------+
|    50|79087.96|
+------+--------+



In [136]:
df_emp.agg(
    min("salary").alias("최소급여"),
    max("salary").alias("최대급여"),
    sum("salary").alias("총급여")
).show()

+--------+--------+-------+
|최소급여|최대급여| 총급여|
+--------+--------+-------+
|   41016|  118953|3954398|
+--------+--------+-------+



In [137]:
df_emp.agg(countDistinct("hire_year").alias("입사년도수")).show()


+----------+
|입사년도수|
+----------+
|         5|
+----------+



### 그룹별 집계 (groupBy + agg)

In [138]:
# 부서별 통계
dept_stats = df_emp.groupBy("department").agg(
    count("*").alias("인원수"),                           # 부서별 인원
    spark_round(avg("salary"), 2).alias("평균급여"),      # 평균 급여 (소수 2자리)
    min("salary").alias("최소급여"),                      # 최소 급여
    max("salary").alias("최대급여"),                      # 최대 급여
    sum("salary").alias("총급여"),                        # 급여 합계
)

print("=== 부서별 통계 ===")
dept_stats.orderBy(col("인원수").desc()).show()

=== 부서별 통계 ===
+-----------+------+--------+--------+--------+-------+
| department|인원수|평균급여|최소급여|최대급여| 총급여|
+-----------+------+--------+--------+--------+-------+
|         HR|    13|85255.23|   51534|  112409|1108318|
|  Marketing|    10| 76457.5|   42568|  117189| 764575|
|      Sales|    10| 70509.7|   42695|  111211| 705097|
|    Finance|    10| 82034.8|   41016|  118953| 820348|
|Engineering|     7|79437.14|   45258|  107563| 556060|
+-----------+------+--------+--------+--------+-------+



In [139]:
# -----------------------------------------------------------------------------
# groupBy(): 여러 컬럼으로 그룹화
# -----------------------------------------------------------------------------

# 여러 컬럼을 쉼표로 나열하면 조합별로 그룹화
# (부서, 입사년도) 조합별 통계

dept_year_stats = df_emp.groupBy("department", "hire_year").agg(
    count("*").alias("인원수"),
    spark_round(avg("salary"), 0).alias("평균급여"),
)

print("=== 부서 + 입사년도별 통계 ===")
dept_year_stats.orderBy("department", "hire_year").show(15)

=== 부서 + 입사년도별 통계 ===
+-----------+---------+------+--------+
| department|hire_year|인원수|평균급여|
+-----------+---------+------+--------+
|Engineering|     2020|     3| 57602.0|
|Engineering|     2022|     2| 90085.0|
|Engineering|     2023|     1|101858.0|
|Engineering|     2024|     1|101228.0|
|    Finance|     2020|     1| 41016.0|
|    Finance|     2021|     1|118953.0|
|    Finance|     2022|     3| 82663.0|
|    Finance|     2023|     3| 85192.0|
|    Finance|     2024|     2| 78407.0|
|         HR|     2020|     4| 86352.0|
|         HR|     2021|     1| 51534.0|
|         HR|     2022|     1| 78044.0|
|         HR|     2023|     3|102099.0|
|         HR|     2024|     4| 81759.0|
|  Marketing|     2020|     3| 73003.0|
+-----------+---------+------+--------+
only showing top 15 rows



In [140]:
# 부서별 인원 수 (간단 버전)
df_emp.groupBy("department").count().show()

# 부서별 급여 합계 (간단 버전)
df_emp.groupBy("department").sum("salary").show()

# 부서별 평균 급여 (간단 버전)
df_emp.groupBy("department").avg("salary").show()

+-----------+-----+
| department|count|
+-----------+-----+
|         HR|   13|
|  Marketing|   10|
|      Sales|   10|
|Engineering|    7|
|    Finance|   10|
+-----------+-----+

+-----------+-----------+
| department|sum(salary)|
+-----------+-----------+
|         HR|    1108318|
|  Marketing|     764575|
|      Sales|     705097|
|Engineering|     556060|
|    Finance|     820348|
+-----------+-----------+

+-----------+-----------------+
| department|      avg(salary)|
+-----------+-----------------+
|         HR|85255.23076923077|
|  Marketing|          76457.5|
|      Sales|          70509.7|
|Engineering|79437.14285714286|
|    Finance|          82034.8|
+-----------+-----------------+



In [141]:
# 조건부 집계 : when

# 부서별 고연봉자(8만 이상) 수
conditional_count = df_emp.groupBy("department").agg(
    count("*").alias("전체인원"),
    # when 조건이 참인 경우만 카운트
    count(when(col("salary") >= 80000, 1)).alias("고연봉자수"),
    count(when(col("salary") < 50000, 1)).alias("저연봉자수"),
    count(when(col("age") >= 40, 1)).alias("40대 이상")
)

print("=== 조건부 집계 ===")
conditional_count.show()

=== 조건부 집계 ===
+-----------+--------+----------+----------+---------+
| department|전체인원|고연봉자수|저연봉자수|40대 이상|
+-----------+--------+----------+----------+---------+
|         HR|      13|         7|         0|        5|
|  Marketing|      10|         5|         3|        3|
|      Sales|      10|         4|         2|        4|
|Engineering|       7|         3|         1|        4|
|    Finance|      10|         5|         2|        4|
+-----------+--------+----------+----------+---------+



### 실습 2

In [144]:
# 부서별 직원 수를 조회
df_emp.groupBy("department").count().show()

+-----------+-----+
| department|count|
+-----------+-----+
|         HR|   13|
|  Marketing|   10|
|      Sales|   10|
|Engineering|    7|
|    Finance|   10|
+-----------+-----+



In [150]:
# 부서별로 평균 급여와 최고 급여를 조회
df_emp.groupBy("department").agg(
    avg("salary").cast("int").alias("평균급여"),
    max("salary").alias("최고급여")
).show()

+-----------+--------+--------+
| department|평균급여|최고급여|
+-----------+--------+--------+
|         HR|   85255|  112409|
|  Marketing|   76457|  117189|
|      Sales|   70509|  111211|
|Engineering|   79437|  107563|
|    Finance|   82034|  118953|
+-----------+--------+--------+



In [ ]:
# hire_year별로 입사한 직원 수를 조회하고, 입사년도 순으로 정렬
df_emp.groupBy("hire_year").count().orderBy("hire_year").show()

+---------+-----+
|hire_year|count|
+---------+-----+
|     2020|   13|
|     2021|    6|
|     2022|   10|
|     2023|   13|
|     2024|    8|
+---------+-----+



In [154]:
# 부서와 입사년도 조합별 인원수를 조회
df_emp.groupBy("department","hire_year").count().orderBy("department","hire_year").show()

+-----------+---------+-----+
| department|hire_year|count|
+-----------+---------+-----+
|Engineering|     2020|    3|
|Engineering|     2022|    2|
|Engineering|     2023|    1|
|Engineering|     2024|    1|
|    Finance|     2020|    1|
|    Finance|     2021|    1|
|    Finance|     2022|    3|
|    Finance|     2023|    3|
|    Finance|     2024|    2|
|         HR|     2020|    4|
|         HR|     2021|    1|
|         HR|     2022|    1|
|         HR|     2023|    3|
|         HR|     2024|    4|
|  Marketing|     2020|    3|
|  Marketing|     2021|    2|
|  Marketing|     2022|    3|
|  Marketing|     2023|    1|
|  Marketing|     2024|    1|
|      Sales|     2020|    2|
+-----------+---------+-----+
only showing top 20 rows



### 특수 집계 함수

In [157]:
# -----------------------------------------------------------------------------
# collect_list(), collect_set(): 값들을 배열로 모으기
# -----------------------------------------------------------------------------

# collect_list(): 그룹의 모든 값을 배열로 수집 (중복 포함)
# collect_set(): 그룹의 고유 값만 배열로 수집 (중복 제거)

# 부서별 직원 이름 목록
name_list = df_emp.groupBy("department").agg(
    collect_list("name").alias("직원목록"),      # 모든 이름
    collect_set("hire_year").alias("입사년도"),  # 고유 입사년도
    count("*").alias("인원수"),
)

print("=== 부서별 직원 목록 ===")
name_list.show(truncate=False)

=== 부서별 직원 목록 ===
+-----------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------+------+
|department |직원목록                                                                                                                                                                |입사년도                      |인원수|
+-----------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------+------+
|HR         |[Employee_1, Employee_11, Employee_15, Employee_17, Employee_20, Employee_23, Employee_29, Employee_30, Employee_32, Employee_33, Employee_41, Employee_43, Employee_50]|[2022, 2023, 2020, 2024, 2021]|13    |
|Marketing  |[Employee_3, Employee_7, Employee_8, Employee_9, Employee_12, Employee_26, Employee_27, Employee

In [158]:
# -----------------------------------------------------------------------------
# first(), last(): 첫 번째/마지막 값
# -----------------------------------------------------------------------------

# first(): 그룹의 첫 번째 값 반환
# last(): 그룹의 마지막 값 반환
# 주의: 정렬 없이 사용하면 결과가 비결정적

# 부서별 첫 번째 직원 (정렬 없이는 순서 보장 안 됨)
first_emp = df_emp.groupBy("department").agg(
    first("name").alias("첫번째직원"),
    first("salary").alias("해당급여"),
)

print("=== 부서별 첫 번째 직원 ===")
first_emp.show()

=== 부서별 첫 번째 직원 ===
+-----------+-----------+--------+
| department| 첫번째직원|해당급여|
+-----------+-----------+--------+
|Engineering|Employee_19|   72606|
|    Finance| Employee_2|   88555|
|         HR| Employee_1|   63483|
|  Marketing| Employee_3|   57159|
|      Sales| Employee_6|  109479|
+-----------+-----------+--------+



### 조인 (Join)

In [159]:
# -----------------------------------------------------------------------------
# 조인 데이터 확인
# -----------------------------------------------------------------------------

print("=== 직원 데이터 ===")
df_emp.select("emp_id", "name", "department").show(5)

print("=== 부서 데이터 ===")
df_dept.show()

=== 직원 데이터 ===
+------+----------+----------+
|emp_id|      name|department|
+------+----------+----------+
|  E001|Employee_1|        HR|
|  E002|Employee_2|   Finance|
|  E003|Employee_3| Marketing|
|  E004|Employee_4|   Finance|
|  E005|Employee_5|   Finance|
+------+----------+----------+
only showing top 5 rows

=== 부서 데이터 ===
+-----------+---------+------+--------+
|  dept_name|dept_head|budget|location|
+-----------+---------+------+--------+
|Engineering|    Alice|500000|   Seoul|
|      Sales|      Bob|300000|   Busan|
|  Marketing|  Charlie|200000|   Seoul|
|         HR|    Diana|150000|   Daegu|
|    Finance|      Eve|400000|   Seoul|
|      Legal|    Frank|100000| Incheon|
+-----------+---------+------+--------+



In [160]:
# -----------------------------------------------------------------------------
# Inner Join: 양쪽에 모두 있는 것만
# -----------------------------------------------------------------------------

# join(df2, 조건, how="inner")
# - df2: 조인할 DataFrame
# - 조건: 조인 키 (문자열 또는 조건식)
# - how: 조인 타입 (기본값 "inner")

# 직원과 부서 정보 조인
# department(직원) == dept_name(부서)인 행만 결합
df_joined = df_emp.join(
    df_dept,                                    # 조인할 DataFrame
    df_emp.department == df_dept.dept_name,    # 조인 조건
    "inner"                                     # 조인 타입
)

print("=== Inner Join 결과 ===")
df_joined.select(
    "name", "department", "salary", "dept_head", "location"
).show(10)

=== Inner Join 결과 ===
+-----------+----------+------+---------+--------+
|       name|department|salary|dept_head|location|
+-----------+----------+------+---------+--------+
| Employee_1|        HR| 63483|    Diana|   Daegu|
| Employee_2|   Finance| 88555|      Eve|   Seoul|
| Employee_3| Marketing| 57159|  Charlie|   Seoul|
| Employee_4|   Finance| 75920|      Eve|   Seoul|
| Employee_5|   Finance|107121|      Eve|   Seoul|
| Employee_6|     Sales|109479|      Bob|   Busan|
| Employee_7| Marketing| 59457|  Charlie|   Seoul|
| Employee_8| Marketing|106557|  Charlie|   Seoul|
| Employee_9| Marketing|117189|  Charlie|   Seoul|
|Employee_10|   Finance|118953|      Eve|   Seoul|
+-----------+----------+------+---------+--------+
only showing top 10 rows



In [161]:
# -----------------------------------------------------------------------------
# 조인 키가 같은 이름일 때: 문자열로 지정
# -----------------------------------------------------------------------------

# 조인 키 컬럼명이 같으면 문자열로 간단히 지정
# 결과에서 조인 키 컬럼이 하나만 남음 (중복 제거)

# 데이터 준비: 컬럼명 맞추기
df_dept_renamed = df_dept.withColumnRenamed("dept_name", "department")

# 같은 이름으로 조인
df_simple_join = df_emp.join(
    df_dept_renamed,
    "department",       # 양쪽에 같은 이름의 컬럼
    "inner"
)

print("=== 같은 컬럼명으로 조인 ===")
df_simple_join.select("name", "department", "salary", "dept_head").show(5)

=== 같은 컬럼명으로 조인 ===
+----------+----------+------+---------+
|      name|department|salary|dept_head|
+----------+----------+------+---------+
|Employee_1|        HR| 63483|    Diana|
|Employee_2|   Finance| 88555|      Eve|
|Employee_3| Marketing| 57159|  Charlie|
|Employee_4|   Finance| 75920|      Eve|
|Employee_5|   Finance|107121|      Eve|
+----------+----------+------+---------+
only showing top 5 rows



In [162]:
# -----------------------------------------------------------------------------
# Left Join: 왼쪽 전체 + 오른쪽 매칭
# -----------------------------------------------------------------------------

# left join: 왼쪽 DataFrame의 모든 행 유지
# 오른쪽에 매칭되는 행이 없으면 NULL

# IT 부서 직원 추가 (df_dept에는 IT 부서 없음)
df_emp_with_it = df_emp.union(
    spark.createDataFrame([
        ("E999", "IT_Person", "IT", 70000, 30, 2024)
    ], df_emp.columns)
)

# Left Join: 모든 직원 + 부서 정보 (없으면 NULL)
df_left = df_emp_with_it.join(
    df_dept,
    df_emp_with_it.department == df_dept.dept_name,
    "left"
)

print("=== Left Join 결과 (IT 부서 직원 포함) ===")
df_left.filter(col("dept_name").isNull()).select(
    "name", "department", "dept_name", "dept_head"
).show()

=== Left Join 결과 (IT 부서 직원 포함) ===


+---------+----------+---------+---------+
|     name|department|dept_name|dept_head|
+---------+----------+---------+---------+
|IT_Person|        IT|     NULL|     NULL|
+---------+----------+---------+---------+



### 실습


In [163]:
# Inner Join
df_emp.join(
    df_dept,
    df_emp.department == df_dept.dept_name,
    "inner"
).select("name","department","dept_head").show(10)

+-----------+----------+---------+
|       name|department|dept_head|
+-----------+----------+---------+
| Employee_1|        HR|    Diana|
| Employee_2|   Finance|      Eve|
| Employee_3| Marketing|  Charlie|
| Employee_4|   Finance|      Eve|
| Employee_5|   Finance|      Eve|
| Employee_6|     Sales|      Bob|
| Employee_7| Marketing|  Charlie|
| Employee_8| Marketing|  Charlie|
| Employee_9| Marketing|  Charlie|
|Employee_10|   Finance|      Eve|
+-----------+----------+---------+
only showing top 10 rows



In [ ]:
# Left Join
df_emp.join(
    df_dept,
    df_emp.department == df_dept.dept_name,
    "left"
).select("name","department","dept_head", "location").show(10)

+-----------+----------+---------+--------+
|       name|department|dept_head|location|
+-----------+----------+---------+--------+
| Employee_1|        HR|    Diana|   Daegu|
| Employee_2|   Finance|      Eve|   Seoul|
| Employee_3| Marketing|  Charlie|   Seoul|
| Employee_4|   Finance|      Eve|   Seoul|
| Employee_5|   Finance|      Eve|   Seoul|
| Employee_6|     Sales|      Bob|   Busan|
| Employee_7| Marketing|  Charlie|   Seoul|
| Employee_8| Marketing|  Charlie|   Seoul|
| Employee_9| Marketing|  Charlie|   Seoul|
|Employee_10|   Finance|      Eve|   Seoul|
+-----------+----------+---------+--------+
only showing top 10 rows



In [166]:
# 같은 컬럼명으로 조인
df_renamed = df_dept.withColumnRenamed("dept_name","department")

df_emp.join(df_renamed, "department", "inner").show(5)


+----------+------+----------+------+---+---------+---------+------+--------+
|department|emp_id|      name|salary|age|hire_year|dept_head|budget|location|
+----------+------+----------+------+---+---------+---------+------+--------+
|        HR|  E001|Employee_1| 63483| 50|     2024|    Diana|150000|   Daegu|
|   Finance|  E002|Employee_2| 88555| 33|     2022|      Eve|400000|   Seoul|
| Marketing|  E003|Employee_3| 57159| 52|     2024|  Charlie|200000|   Seoul|
|   Finance|  E004|Employee_4| 75920| 31|     2023|      Eve|400000|   Seoul|
|   Finance|  E005|Employee_5|107121| 33|     2024|      Eve|400000|   Seoul|
+----------+------+----------+------+---+---------+---------+------+--------+
only showing top 5 rows



In [ ]:
### Broadcast Join


### 피벗 (Pivot)

In [4]:
df_sales.show(5)

+----------+-------+------+------+--------+
|      date|product|region|amount|quantity|
+----------+-------+------+------+--------+
|2026-01-01|      A| Seoul|   224|       5|
|2026-01-02|      A| Daegu|   249|       4|
|2026-01-03|      C| Daegu|   413|       2|
|2026-01-04|      A| Busan|   669|      10|
|2026-01-05|      B| Daegu|   441|      19|
+----------+-------+------+------+--------+
only showing top 5 rows



In [169]:
# 매출 데이터로 피벗 테이블 생성
# 지역별, 제품별 매출 합계
pivot_sales = df_sales.groupBy("region").pivot(
    "Product",
    ["A","B","C"]
).sum("amount")

pivot_sales.show()

+------+----+----+----+
|region|   A|   B|   C|
+------+----+----+----+
| Busan|7027|7184|7879|
| Daegu|5173|5223|5269|
| Seoul|6312|5536|6269|
+------+----+----+----+



In [5]:
# 피벗 + 여러 집계 함수
# agg() 안에 여러 집계 지정

pivot_multi = df_sales.groupBy("region").pivot(
    "product", ["A", "B", "C"]
).agg(
    sum("amount").alias("총매출"),
    avg("amount").alias("평균매출"),
)

print("=== 피벗 + 여러 집계 ===")
pivot_multi.show()

=== 피벗 + 여러 집계 ===
+------+--------+-----------------+--------+-----------------+--------+-----------------+
|region|A_총매출|       A_평균매출|B_총매출|       B_평균매출|C_총매출|       C_평균매출|
+------+--------+-----------------+--------+-----------------+--------+-----------------+
| Busan|    7027|638.8181818181819|    7184|552.6153846153846|    7879|656.5833333333334|
| Daegu|    5173|431.0833333333333|    5223|            522.3|    5269|            479.0|
| Seoul|    6312|            526.0|    5536|790.8571428571429|    6269|522.4166666666666|
+------+--------+-----------------+--------+-----------------+--------+-----------------+



### 실습

In [7]:
# df_sales에서 지역(region)별로 제품(product)을 피벗하여 매출(amount) 합계를 조회
df_sales.groupBy("region").pivot(
    "product",
    ["A","B","C"]
).sum("amount").show(5)

+------+----+----+----+
|region|   A|   B|   C|
+------+----+----+----+
| Busan|7027|7184|7879|
| Daegu|5173|5223|5269|
| Seoul|6312|5536|6269|
+------+----+----+----+



In [10]:
# 피벗 시 product 값을 ["A", "B"]만 지정하여 조회
df_sales.groupBy("region").pivot("product", ["A","B"]).sum("amount").show(5)

+------+----+----+
|region|   A|   B|
+------+----+----+
| Busan|7027|7184|
| Daegu|5173|5223|
| Seoul|6312|5536|
+------+----+----+



In [13]:
# 지역별로 제품을 피벗하여 평균 매출을 조회
df_sales.groupBy("region").pivot(
    "product",
    ["A","B","C"]
).agg(
    avg("amount")
).show(5)

+------+-----------------+-----------------+-----------------+
|region|                A|                B|                C|
+------+-----------------+-----------------+-----------------+
| Busan|638.8181818181819|552.6153846153846|656.5833333333334|
| Daegu|431.0833333333333|            522.3|            479.0|
| Seoul|            526.0|790.8571428571429|522.4166666666666|
+------+-----------------+-----------------+-----------------+



### Spark SQL 연동

In [15]:
# -----------------------------------------------------------------------------
# createOrReplaceTempView(): SQL 테이블 등록
# -----------------------------------------------------------------------------
#
# createOrReplaceTempView(이름): 임시 뷰로 등록
# - 해당 SparkSession 내에서만 유효
# - 세션 종료 시 자동 삭제
# - 같은 이름의 뷰가 있으면 덮어쓰기 (Replace)

# DataFrame을 SQL 테이블(뷰)로 등록
df_emp.createOrReplaceTempView("employees")
df_dept.createOrReplaceTempView("departments")

print("SQL 테이블 등록 완료: employees, departments")

# -----------------------------------------------------------------------------
# 등록 후 사용 예시
# -----------------------------------------------------------------------------
# 이제 SQL 문법으로 DataFrame을 조회할 수 있습니다!
# spark.sql("SELECT * FROM employees WHERE salary > 5000")
# spark.sql("SELECT dept_id, AVG(salary) FROM employees GROUP BY dept_id")

SQL 테이블 등록 완료: employees, departments


In [16]:
# -----------------------------------------------------------------------------
# spark.sql(): SQL 쿼리 실행
# -----------------------------------------------------------------------------

# spark.sql(쿼리문): SQL 실행 후 DataFrame 반환
# 복잡한 로직을 SQL로 작성 가능

# SQL로 부서별 통계
sql_result = spark.sql("""
    SELECT
        department,
        COUNT(*) as `인원수`,
        ROUND(AVG(salary), 2) as `평균급여`,
        MAX(salary) as `최고급여`
    FROM employees
    GROUP BY department
    ORDER BY `인원수` DESC
""")

print("=== SQL 쿼리 결과 ===")
sql_result.show()

=== SQL 쿼리 결과 ===
+-----------+------+--------+--------+
| department|인원수|평균급여|최고급여|
+-----------+------+--------+--------+
|         HR|    13|85255.23|  112409|
|  Marketing|    10| 76457.5|  117189|
|      Sales|    10| 70509.7|  111211|
|    Finance|    10| 82034.8|  118953|
|Engineering|     7|79437.14|  107563|
+-----------+------+--------+--------+



In [19]:
# -----------------------------------------------------------------------------
# SQL: 조인
# -----------------------------------------------------------------------------

# SQL로 조인 쿼리
sql_join = spark.sql("""
    SELECT
        e.name,
        e.department,
        e.salary,
        d.dept_head,
        d.location
    FROM employees e
    JOIN departments d ON e.department = d.dept_name
    WHERE e.salary >= 70000
    ORDER BY e.salary DESC
""")

print("=== SQL 조인 결과 ===")
sql_join.show(10)

=== SQL 조인 결과 ===
+-----------+-----------+------+---------+--------+
|       name| department|salary|dept_head|location|
+-----------+-----------+------+---------+--------+
|Employee_10|    Finance|118953|      Eve|   Seoul|
| Employee_9|  Marketing|117189|  Charlie|   Seoul|
|Employee_26|  Marketing|114065|  Charlie|   Seoul|
|Employee_15|         HR|112409|    Diana|   Daegu|
|Employee_16|      Sales|111211|      Bob|   Busan|
| Employee_6|      Sales|109479|      Bob|   Busan|
|Employee_39|Engineering|107563|    Alice|   Seoul|
| Employee_5|    Finance|107121|      Eve|   Seoul|
| Employee_8|  Marketing|106557|  Charlie|   Seoul|
|Employee_17|         HR|105697|    Diana|   Daegu|
+-----------+-----------+------+---------+--------+
only showing top 10 rows



In [20]:
# -----------------------------------------------------------------------------
# expr(): SQL 표현식을 DataFrame에서 사용
# -----------------------------------------------------------------------------

# expr(): SQL 표현식을 Column으로 변환
# select(), withColumn() 등에서 SQL 문법 사용 가능

# SQL 표현식으로 새 컬럼 추가
df_with_expr = df_emp.withColumn(
    "salary_grade",
    expr("CASE WHEN salary >= 80000 THEN 'High' ELSE 'Normal' END")
).withColumn(
    "bonus",
    expr("salary * 0.1")  # 급여의 10%
)

print("=== expr() 사용 예시 ===")
df_with_expr.select("name", "salary", "salary_grade", "bonus").show(5)

=== expr() 사용 예시 ===
+----------+------+------------+-------+
|      name|salary|salary_grade|  bonus|
+----------+------+------------+-------+
|Employee_1| 63483|      Normal| 6348.3|
|Employee_2| 88555|        High| 8855.5|
|Employee_3| 57159|      Normal| 5715.9|
|Employee_4| 75920|      Normal| 7592.0|
|Employee_5|107121|        High|10712.1|
+----------+------+------------+-------+
only showing top 5 rows



### 실습

In [22]:
# df_emp를 "emp"라는 이름의 임시 뷰로 등록
df_emp.createOrReplaceTempView("emp")

In [ ]:
# 등록한 임시 뷰 확인 
spark.sql("SELECT * FROM emp").show()


+------+-----------+-----------+------+---+---------+
|emp_id|       name| department|salary|age|hire_year|
+------+-----------+-----------+------+---+---------+
|  E001| Employee_1|         HR| 63483| 50|     2024|
|  E002| Employee_2|    Finance| 88555| 33|     2022|
|  E003| Employee_3|  Marketing| 57159| 52|     2024|
|  E004| Employee_4|    Finance| 75920| 31|     2023|
|  E005| Employee_5|    Finance|107121| 33|     2024|
|  E006| Employee_6|      Sales|109479| 32|     2022|
|  E007| Employee_7|  Marketing| 59457| 36|     2022|
|  E008| Employee_8|  Marketing|106557| 26|     2023|
|  E009| Employee_9|  Marketing|117189| 25|     2021|
|  E010|Employee_10|    Finance|118953| 40|     2021|
|  E011|Employee_11|         HR| 92995| 47|     2024|
|  E012|Employee_12|  Marketing| 80757| 47|     2020|
|  E013|Employee_13|    Finance| 49692| 54|     2024|
|  E014|Employee_14|      Sales| 85758| 48|     2023|
|  E015|Employee_15|         HR|112409| 29|     2023|
|  E016|Employee_16|      Sa

In [25]:
# SQL로 employees 테이블에서 salary가 70000 이상인 직원의 name과 salary를 조회
spark.sql("""
    SELECT name, salary
    FROM employees
    WHERE salary >= 70000
""").show()

+-----------+------+
|       name|salary|
+-----------+------+
| Employee_2| 88555|
| Employee_4| 75920|
| Employee_5|107121|
| Employee_6|109479|
| Employee_8|106557|
| Employee_9|117189|
|Employee_10|118953|
|Employee_11| 92995|
|Employee_12| 80757|
|Employee_14| 85758|
|Employee_15|112409|
|Employee_16|111211|
|Employee_17|105697|
|Employee_18| 77065|
|Employee_19| 72606|
|Employee_21| 80397|
|Employee_23| 95591|
|Employee_26|114065|
|Employee_29|103704|
|Employee_30| 79099|
+-----------+------+
only showing top 20 rows



In [26]:
# SQL로 부서별 평균 급여를 조회
spark.sql("""
    SELECT department, AVG(salary) as avg_salary
    FROM employees
    GROUP BY department
"""
).show()

+-----------+-----------------+
| department|       avg_salary|
+-----------+-----------------+
|         HR|85255.23076923077|
|  Marketing|          76457.5|
|      Sales|          70509.7|
|Engineering|79437.14285714286|
|    Finance|          82034.8|
+-----------+-----------------+



In [27]:
# expr()을 사용하여 salary의 5%를 "bonus" 컬럼으로 추가
df_emp.withColumn("bonus", expr("salary * 0.05")).select("name","salary","bonus").show(5)

+----------+------+-------+
|      name|salary|  bonus|
+----------+------+-------+
|Employee_1| 63483|3174.15|
|Employee_2| 88555|4427.75|
|Employee_3| 57159|2857.95|
|Employee_4| 75920|3796.00|
|Employee_5|107121|5356.05|
+----------+------+-------+
only showing top 5 rows



### 과제: 영업팀 월간 실적 대시보드
(해야함)

In [29]:
# Step 1: 직원-부서 조인
df_step1 = df_emp.join(
    df_dept,
    df_emp.department == df_dept.dept_name,
    "inner"
).select(
    "name","department","salary","dept_head","location","budget"
    )

print("=== 직원-부서 통합 데이터 ===")
df_step1.show(10)
print(f"조인 결과: {df_step1.count()}명")

=== 직원-부서 통합 데이터 ===
+-----------+----------+------+---------+--------+------+
|       name|department|salary|dept_head|location|budget|
+-----------+----------+------+---------+--------+------+
| Employee_1|        HR| 63483|    Diana|   Daegu|150000|
| Employee_2|   Finance| 88555|      Eve|   Seoul|400000|
| Employee_3| Marketing| 57159|  Charlie|   Seoul|200000|
| Employee_4|   Finance| 75920|      Eve|   Seoul|400000|
| Employee_5|   Finance|107121|      Eve|   Seoul|400000|
| Employee_6|     Sales|109479|      Bob|   Busan|300000|
| Employee_7| Marketing| 59457|  Charlie|   Seoul|200000|
| Employee_8| Marketing|106557|  Charlie|   Seoul|200000|
| Employee_9| Marketing|117189|  Charlie|   Seoul|200000|
|Employee_10|   Finance|118953|      Eve|   Seoul|400000|
+-----------+----------+------+---------+--------+------+
only showing top 10 rows

조인 결과: 50명


In [33]:
# Step 2: 지역별/제품별 매출 집계
df_step2_region = df_sales.groupBy("region").agg(
    sum("amount").alias("총매출"), 
    spark_round(avg("amount"), 2).alias("평균매출"), 
    count("*").alias("거래건수")
).orderBy(col("총매출").desc())

print("=== 지역별 매출 현황 ===")
df_step2_region.show()

df_step2_product = df_sales.groupBy("product").agg(
    sum("amount").alias("총매출"),
    max("amount").alias("최대거래액")
).orderBy(col("총매출").desc())

print("=== 제품별 매출 현황 ===")
df_step2_product.show()

=== 지역별 매출 현황 ===
+------+------+--------+--------+
|region|총매출|평균매출|거래건수|
+------+------+--------+--------+
| Daegu| 20263|  562.86|      36|
| Seoul| 18991|  633.03|      30|
| Busan| 12329|  513.71|      24|
+------+------+--------+--------+

=== 제품별 매출 현황 ===
+-------+------+----------+
|product|총매출|최대거래액|
+-------+------+----------+
|      B| 18579|       996|
|      A| 17020|       878|
|      C| 15984|       980|
+-------+------+----------+



In [34]:
# Step 3: 조건부 집계 - 목표 달성 분석
target_analysis = df_sales.groupBy("region").agg(
    count("*").alias("전체건수"),
    count(when(col("amount") >= 500, 1)).alias("달성건수"),
    count(when(col("amount") < 500, 1)).alias("미달건수"),
    spark_round(
        count(when(col("amount") >= 500, 1)) / count("*") * 100, 1
    ).alias("달성률")
)

print("=== 목표 달성 분석 (목표: 500) ===")
target_analysis.orderBy(col("달성률").desc()).show()

=== 목표 달성 분석 (목표: 500) ===
+------+--------+--------+--------+------+
|region|전체건수|달성건수|미달건수|달성률|
+------+--------+--------+--------+------+
| Seoul|      30|      19|      11|  63.3|
| Busan|      24|      14|      10|  58.3|
| Daegu|      36|      21|      15|  58.3|
+------+--------+--------+--------+------+



In [36]:
# 피벗 테이블 생성
pivot_table = df_sales.groupBy("region").pivot(
    "product", ["A","B","C"]
).sum("amount")

print("=== 지역 × 제품 매출 피벗 테이블 ===")
pivot_table.show()

=== 지역 × 제품 매출 피벗 테이블 ===
+------+----+----+----+
|region|   A|   B|   C|
+------+----+----+----+
| Busan|4688|4889|2752|
| Daegu|7370|5918|6975|
| Seoul|4962|7772|6257|
+------+----+----+----+



In [37]:
# SQL로 종합 분석
# 테이블 등록
df_sales.createOrReplaceTempView("sales")
df_emp.createOrReplaceTempView("employees")

# 지역별 매출 Top 3
print("=== [SQL] 지역별 매출 Top 3 ===")
spark.sql("""
    SELECT region, SUM(amount) as total_sales
    FROM sales
    GROUP BY region
    ORDER BY total_sales DESC
    LIMIT 3
""").show()

# 평균 매출 400 이상 제품
print("=== [SQL] 평균 매출 400 이상 제품 ===")
spark.sql("""
    SELECT product, AVG(amount) as avg_sales
    FROM sales
    GROUP BY product
    HAVING AVG(amount) >= 400
    ORDER BY avg_sales DESC
""").show()

=== [SQL] 지역별 매출 Top 3 ===
+------+-----------+
|region|total_sales|
+------+-----------+
| Daegu|      20263|
| Seoul|      18991|
| Busan|      12329|
+------+-----------+

=== [SQL] 평균 매출 400 이상 제품 ===
+-------+-----------------+
|product|        avg_sales|
+-------+-----------------+
|      B|599.3225806451613|
|      C|570.8571428571429|
|      A|549.0322580645161|
+-------+-----------------+



### 실무 핵심 패턴

In [17]:
# -----------------------------------------------------------------------------
# 환경 설정
# -----------------------------------------------------------------------------
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, when,
    # NULL 처리
    coalesce, isnan,
    # 문자열 함수
    concat, concat_ws, substring, length, trim, ltrim, rtrim,
    upper, lower, initcap, regexp_replace, regexp_extract, split,
    lpad, rpad,
    # 날짜/시간 함수
    current_date, current_timestamp, to_date, to_timestamp, date_format,
    year, month, dayofmonth, dayofweek, hour, minute,
    date_add, date_sub, datediff, months_between, trunc,
    # 집계/윈도우
    count, sum, avg, min, max, first, last,
    row_number, rank, dense_rank, lag, lead,
    # 기타
    round as spark_round, expr,
)
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
import pandas as pd
import numpy as np
import os

# SparkSession 생성
spark = SparkSession.builder \
    .appName("PySpark-Advanced-Patterns") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", 10) \
    .getOrCreate()

os.makedirs("/tmp/spark_tutorial", exist_ok=True)
np.random.seed(42)

# -----------------------------------------------------------------------------
# 테스트 데이터: 다양한 상황을 포함한 직원 데이터
# -----------------------------------------------------------------------------
employees = pd.DataFrame({
    "emp_id": [f"E{i:03d}" for i in range(1, 31)],
    "name": [f"  Employee {i}  " for i in range(1, 31)],  # 앞뒤 공백
    "email": [f"emp{i}@company.com" for i in range(1, 31)],
    "department": np.random.choice(
        ["Engineering", "Sales", "Marketing", None], 30
    ),
    "salary": [
        50000, None, 70000, 80000, None,  # NULL 포함
        60000, 90000, 55000, None, 75000,
        65000, 85000, None, 72000, 68000,
        None, 95000, 62000, 78000, None,
        58000, 82000, 67000, None, 73000,
        69000, 88000, None, 76000, 71000
    ],
    "join_date": pd.date_range("2020-01-15", periods=30, freq="45D").strftime("%Y-%m-%d"),
})
employees.to_csv("/tmp/spark_tutorial/employees_advanced.csv", index=False)

# 매출 시계열 데이터
sales = pd.DataFrame({
    "date": pd.date_range("2026-01-01", periods=90, freq="D").strftime("%Y-%m-%d"),
    "product": np.random.choice(["A", "B", "C"], 90),
    "region": np.random.choice(["Seoul", "Busan", "Daegu"], 90),
    "amount": np.random.randint(100, 1000, 90),
})
sales.to_csv("/tmp/spark_tutorial/sales_ts.csv", index=False)

# DataFrame 로드
df = spark.read.csv("/tmp/spark_tutorial/employees_advanced.csv", header=True, inferSchema=True)
df_sales = spark.read.csv("/tmp/spark_tutorial/sales_ts.csv", header=True, inferSchema=True)

print("데이터 로드 완료!")
print(f"직원: {df.count()}명, 매출: {df_sales.count()}건")
df.show(10)

26/01/20 07:07:16 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


데이터 로드 완료!
직원: 30명, 매출: 90건
+------+---------------+-----------------+-----------+-------+----------+
|emp_id|           name|            email| department| salary| join_date|
+------+---------------+-----------------+-----------+-------+----------+
|  E001|   Employee 1  | emp1@company.com|  Marketing|50000.0|2020-01-15|
|  E002|   Employee 2  | emp2@company.com|       NULL|   NULL|2020-02-29|
|  E003|   Employee 3  | emp3@company.com|Engineering|70000.0|2020-04-14|
|  E004|   Employee 4  | emp4@company.com|  Marketing|80000.0|2020-05-29|
|  E005|   Employee 5  | emp5@company.com|  Marketing|   NULL|2020-07-13|
|  E006|   Employee 6  | emp6@company.com|       NULL|60000.0|2020-08-27|
|  E007|   Employee 7  | emp7@company.com|Engineering|90000.0|2020-10-11|
|  E008|   Employee 8  | emp8@company.com|Engineering|55000.0|2020-11-25|
|  E009|   Employee 9  | emp9@company.com|  Marketing|   NULL|2021-01-09|
|  E010|  Employee 10  |emp10@company.com|      Sales|75000.0|2021-02-23|
+------+--

# NULL 처리(결측치 처리)

In [18]:
# NULL 처리(결측치 처리)